# ETL — Online Retail II → Data Warehouse
### Python · Pandas · SQL Server

**Autor:** Aaron Alejandro Kiwaki Alvarez  
**Fuente:** [Kaggle — Online Retail II UCI](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)

---

## Descripción

Este notebook ejecuta el proceso ETL completo:

| Fase | Descripción |
|---|---|
| **E — Extract** | Lectura del CSV original con 1,067,371 transacciones |
| **T — Transform** | Limpieza y construcción del esquema estrella (5 tablas) |
| **L — Load** | Carga del Data Warehouse en SQL Server |

## Problemas que resuelve el ETL

| Problema | Cantidad | Solución |
|---|---|---|
| `Customer ID` nulos | 243,007 (22.77%) | Asignar cliente `UNKNOWN` |
| Duplicados exactos | 34,335 | Eliminados |
| Cantidades negativas | 22,950 | Excluidas (devoluciones) |
| Precios en cero | 6,202 | Excluidos |
| Códigos especiales (POST, DOT) | 5985 | Excluidos |
| Cancelaciones (Invoice con C) | 19,494 | Marcadas con flag `is_cancelled = 1` |


## Configuración
Ajusta estas variables antes de ejecutar.

In [ ]:
import os

# VS Code detecta automaticamente la carpeta donde estas trabajando
current_dir = os.getcwd()

# Construye la ruta uniendo la carpeta actual con el nombre del archivo
CSV_PATH  = os.path.join(current_dir, 'online_retail_II.csv')

#Credenciales de Base de Datos
DB_SERVER = 'localhost'
DB_NAME   = 'OnlineRetail_DW'
CONN_STR  = (
    f'mssql+pyodbc://{DB_SERVER}/{DB_NAME}'
    '?driver=ODBC+Driver+17+for+SQL+Server'
    '&trusted_connection=yes'
)
print('Configuracion lista.')
print(f'  CSV     : {CSV_PATH}')
print(f'  Servidor: {DB_SERVER}')
print(f'  Base    : {DB_NAME}')

## Librerías

In [55]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text #Puente entre python y bases relacionales
import warnings #modulo que maneja alertas
warnings.filterwarnings('ignore') #quita mensaje de errores menores
print(f'pandas: {pd.__version__}')
print('Librerias cargadas correctamente.')

pandas: 2.3.3
Librerias cargadas correctamente.


---
## STEP 1 — EXTRACT
Lectura del CSV. Se fuerzan tipos en columnas clave para evitar conversiones incorrectas.

In [56]:
df_raw = pd.read_csv(
    CSV_PATH,
    encoding='utf-8',   #lee usando el estandar internacional de caracteres
    low_memory=False,   #Lee toda la columna completa
    dtype={             #Fuerza el tipo de dato en columnas especificas
        'Invoice'       : str,
        'StockCode'     : str,
        'Customer ID'   : str,
    }
)
print(f'Filas cargadas  : {len(df_raw):,}')
print(f'Columnas        : {df_raw.columns.tolist()}')
df_raw.head(3)

Filas cargadas  : 1,067,371
Columnas        : ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


## Diagnóstico de calidad
Antes de transformar, inspeccionamos los problemas reales del dataset.

In [57]:
print('=== NULOS POR COLUMNA ===')
nulls = df_raw.isnull().sum() #convierte celda en un valor boleano, true=null, false=con datos.
pct = (nulls/len(df_raw)*100).round(2) #total de nulos/total de filas*100
for col in df_raw.columns:
    print(f'{col:<15}: {nulls[col]:>7} nulos ({pct[col]}%)')

print(f'\n=== OTROS PROBLEMAS ===')
print(f' Duplicados exactos         : {df_raw.duplicated().sum():>7,}')
print(f' Factucas canceladas (C)    : {df_raw["Invoice"].str.startswith("C").sum():>7,}')
print(f' Quantity negatica          : {(pd.to_numeric(df_raw["Quantity"], errors="coerce") < 0).sum():>7,}')
print(f' Price igual a 0            : {(pd.to_numeric(df_raw["Price"], errors="coerce") == 0).sum():>7}')

=== NULOS POR COLUMNA ===
Invoice        :       0 nulos (0.0%)
StockCode      :       0 nulos (0.0%)
Description    :    4382 nulos (0.41%)
Quantity       :       0 nulos (0.0%)
InvoiceDate    :       0 nulos (0.0%)
Price          :       0 nulos (0.0%)
Customer ID    :  243007 nulos (22.77%)
Country        :       0 nulos (0.0%)

=== OTROS PROBLEMAS ===
 Duplicados exactos         :  34,335
 Factucas canceladas (C)    :  19,494
 Quantity negatica          :  22,950
 Price igual a 0            :    6202


## Validamos StockCode Existentes
Se validan que codigos aparte de los digitos se encuentran en la base de datos.

In [58]:
# 1. Configuración de Pandas: Forzar a mostrar TODAS las filas sin truncar
pd.set_option('display.max_rows', None)

# 2. Filtramos lo que NO sean 5 numeros al inicio
anomalias = df_raw[~df_raw['StockCode'].str.contains(r'^\d{5}', na=False, regex=True)]
# r'^\d{5}' ^(desde el inicio) \d{5}(cuenta 5)
# na=False al hacer la busqueda encuentra un vacio lo transfoma en False

# 3. Extraer la lista unica de sospechosos
codigos_sospechosos = anomalias[['StockCode','Description']].drop_duplicates()

# 4.Obviar (borrar de esta vista) los que empiezan con "DCGS"
codigos_sospechosos = codigos_sospechosos[~codigos_sospechosos['StockCode'].str.startswith('DCGS', na=False)]
codigos_sospechosos = codigos_sospechosos[~codigos_sospechosos['StockCode'].str.startswith('gift', na=False)]
# str nos permite modificar texto

# 5. Imprimir resultados
print(codigos_sospechosos)


           StockCode                          Description
89              POST                              POSTAGE
735                D                             Discount
2379             DOT                       DOTCOM POSTAGE
2697               M                               Manual
9292              C2                             CARRIAGE
18410   BANK CHARGES                         Bank Charges
27994        TEST001              This is a test product.
32207             C2                                  NaN
39411        TEST002              This is a test product.
44614        TEST002                                  NaN
62299           PADS           PADS TO MATCH ALL CUSHIONS
70975         ADJUST  Adjustment by john on 26/01/2010 16
71058         ADJUST  Adjustment by john on 26/01/2010 17
83304           GIFT                                  NaN
96608              m                               Manual
114061             S                              SAMPLES
114180  BANK C

## Validamos Country existentes
Validamos que paises se encuentran en la base de datos

In [59]:
pd.set_option('display.max_rows', None)

conteo_paises = df_raw['Country'].value_counts()
print(conteo_paises)

Country
United Kingdom          981330
EIRE                     17866
Germany                  17624
France                   14330
Netherlands               5140
Spain                     3811
Switzerland               3189
Belgium                   3123
Portugal                  2620
Australia                 1913
Channel Islands           1664
Italy                     1534
Norway                    1455
Sweden                    1364
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    817
Unspecified                756
Greece                     663
Japan                      582
Poland                     535
USA                        535
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                      299
Iceland                    253
Canada                     228
Lithuania                  189
RSA                        169


## Cantidad de paises por cliente
Validamos la cantad de clientes que tienen facturas en diferentes paises


In [60]:
# 1. Agrupar y contar países únicos por cliente
paises_dif = (
    df_raw.groupby('Customer ID')['Country']
    .nunique()
    .reset_index()
    .rename(columns={'Country': 'cantidad_paises'})
)

# 2. Filtrar para ver SOLO a los que tienen 2 o más países diferentes
clientes_viajeros = paises_dif[paises_dif['cantidad_paises'] > 1]

# 3. Mostrar el resultado
print(f"Total de clientes con más de un país: {len(clientes_viajeros)}")
print("\nLista de clientes viajeros:")
print(clientes_viajeros.sort_values(by='cantidad_paises', ascending=False))

Total de clientes con más de un país: 13

Lista de clientes viajeros:
    Customer ID  cantidad_paises
24      12370.0                2
48      12394.0                2
67      12413.0                2
71      12417.0                2
76      12422.0                2
77      12423.0                2
83      12429.0                2
85      12431.0                2
103     12449.0                2
109     12455.0                2
111     12457.0                2
306     12652.0                2
399     12745.0                2


---
## STEP 2 — TRANSFORM
### 2.1 Estandarización de columnas y tipos de dato

In [61]:
df = df_raw.copy()

#Renombrar a snake_Case
df.columns = ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price',
              'customer_id', 'country']

#conversion de tipos
df['invoice_date']  = pd.to_datetime(df['invoice_date'], errors='coerce') #coerce si no coincide = nulo
df['price']         = pd.to_numeric(df['price'], errors ='coerce')
df['quantity']      = pd.to_numeric(df['quantity'], errors='coerce')

# Customer ID: '13085.0' -> '13085', nulos -> 'UNKNOWN'
df['customer_id'] = (
    df['customer_id']
    .fillna('UNKNOWN') # busca vacios y los cambia por UNKNOWN
    .str.replace(r'\.0$', '', regex=True) #remplaza los .0 a ''(vacio)
    .str.strip()  #borra espacios adelanto o al final
)

#Flag de cancelacion
df['is_cancelled'] = df['invoice'].str.startswith('C').astype(int)

# Limpiar descripciones
df['description'] = (
    df['description']
    .fillna('SIN DESCRIPCION') # busca vacios y los cambia por SIN DESCRIPCION
    .str.strip() # borra espacios adelanto o al final
    .str.upper() # vuelve todas las letras a mayusculas
)

print('Tipos despues de la conversion:')
print(df.dtypes)

Tipos despues de la conversion:
invoice                 object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
price                  float64
customer_id             object
country                 object
is_cancelled             int64
dtype: object


### 2.2 Limpieza — eliminación de registros inválidos

In [62]:
filas_antes = len(df)

# 1. Duplicados exactos
df = df.drop_duplicates()
print (f'Duplicados eliminados : {filas_antes - len(df):>7,}')
n = len(df)

# 2. Codigos especiales (no son productos reales)
codigos_especiales = ['POST','DOT','M','D','C2','AMAZONFEE', 'C3',
                      'BANK CHARGES','PADS', 'TEST001', 'TEST002', 
                      'ADJUST', 'ADJUST2', 'm', 'B' ,'CRUK', 'S']
df = df[~df['stock_code'].isin(codigos_especiales)]
print(f'Codigos especiales elim. : {n - len(df):>7,}')
n = len(df)

# 3. Precios invalidos (solo ventas normales, no cancelados)
df = df[~((df['price']<=0)&(df['is_cancelled']==0))]
print(f'Precios invalidos elim.:{n-len(df):>7,}')
n = len(df)

# 4. Cantidades negativas que no son cancelaciones
df = df[~((df['quantity']<=0)&(df['is_cancelled']==0))]
print(f'Cantidades invalidas el.: {n-len(df):>7,}')
n=len(df)

# 5. Fechas nulas
df = df.dropna(subset=['invoice_date'])
print(f'Fechas nulas eliminadas : {n-len(df):>7,}')

# columna calculada: total por linea
df['total_line'] = (df['quantity']*df['price']).round(2)

print(f'\n Filas originales : {filas_antes:>9,}')
print(f' Filas limpias      : {len(df):>9,}')
print(f' Eliminadas         : {filas_antes - len(df):>9,}')

Duplicados eliminados :  34,335
Codigos especiales elim. :   5,719
Precios invalidos elim.:  5,985
Cantidades invalidas el.:       0
Fechas nulas eliminadas :       0

 Filas originales : 1,067,371
 Filas limpias      : 1,021,332
 Eliminadas         :    46,039


### 2.3 Construccion de dimensiones
#### dim_fecha
Atributos temporales derivados de `Invoicedate`, Power bi los usa para crear jerarquias año -> trimestre -> mes -> dia automaticamente.

In [63]:
fechas_unicas = df['invoice_date'].dt.normalize().unique()
# dt(datetime) modificar fechas\normalize modifica hora 00:00:00 \unique extrae lista sin duplicados
# .unique() extrae los valores y los entrega en un formato Array de NumPy(una lista plana de valores sueltos, sin estructura de tabla)
dim_fecha = pd.DataFrame({'fecha_completa': sorted(fechas_unicas)})
# sorted orden de antiguo a nueva| 
dim_fecha['fecha_id']       = range(1,len(dim_fecha)+1)
dim_fecha['anio']           = dim_fecha['fecha_completa'].dt.year
dim_fecha['trimestre']      = dim_fecha['fecha_completa'].dt.quarter
dim_fecha['mes']            = dim_fecha['fecha_completa'].dt.month
dim_fecha['mes_nombre']     = dim_fecha['fecha_completa'].dt.strftime('%B') #String Format Time|traduce el numero del mes a su nombre completo
dim_fecha['semana']         = dim_fecha['fecha_completa'].dt.isocalendar().week.astype(int)
#isocalendar calcula que numero de la semana del año es
dim_fecha['dia']            = dim_fecha['fecha_completa'].dt.day #saca el dia del mes(15)
dim_fecha['dia_semana']     = dim_fecha['fecha_completa'].dt.dayofweek + 1
# pandas cuenta los dias de la semana(considera lunes 0- domingo 6), con esto suma 1(Lunes1-domin7)
dim_fecha['nombre_dia']     = dim_fecha['fecha_completa'].dt.strftime('%A')# traduce el dia a su nombre completo
dim_fecha['es_fin_semana']  = (dim_fecha['dia_semana']>=6).astype(int)
dim_fecha = dim_fecha[['fecha_id','fecha_completa','anio','trimestre',
                       'mes','mes_nombre','semana','dia',
                       'dia_semana','nombre_dia','es_fin_semana']]

print(f'dim_fecha: {len(dim_fecha):,} filas')
dim_fecha.head(3)

dim_fecha: 604 filas


,fecha_id,fecha_completa,anio,trimestre,mes,mes_nombre,semana,dia,dia_semana,nombre_dia,es_fin_semana
0,1,2009-12-01,2009,4,12,December,49,1,2,Tuesday,0
1,2,2009-12-02,2009,4,12,December,49,2,3,Wednesday,0
2,3,2009-12-03,2009,4,12,December,49,3,4,Thursday,0


#### dim_cliente
Segmentacion por frecuencia de compra: UNKNOWN -> NUEVO(1) -> REGULAR (2-5) -> FRECUENTE (6-15) -> VIP (+15)

In [64]:
clientes_unicos = df[['customer_id', 'country']].drop_duplicates(subset='customer_id')

freq = (
    df[df['is_cancelled'] == 0]
    .groupby('customer_id')['invoice']
    .nunique() # Cuenta solo invoice distintas, ignorando productos repetidos del mismo ticket.
    .reset_index()
    .rename(columns = {'invoice': 'num_facturas'})
)
clientes_unicos = clientes_unicos.merge(freq, on='customer_id', how='left')
clientes_unicos['num_facturas']= clientes_unicos['num_facturas'].fillna(0).astype(int)

def segmentar(n):
    if n == 0: return 'UNKNOWN'
    if n == 1: return 'NUEVO'
    if n <=5:  return 'REGULAR'
    if n <=15: return 'FRECUENTE'
    return 'VIP'

clientes_unicos['segmento'] = clientes_unicos['num_facturas'].apply(segmentar)
clientes_unicos['cliente_id'] = range(1, len(clientes_unicos) + 1)
dim_cliente = clientes_unicos [['cliente_id', 'customer_id', 'country', 'segmento', 'num_facturas']]

print(f'dim_clientes: {len(dim_cliente):,} clientes')
print('\nDistribucion por segmento:')
print(dim_cliente['segmento'].value_counts().to_string())
dim_cliente.head(3)

dim_clientes: 5,876 clientes

Distribucion por segmento:
segmento
REGULAR      2448
NUEVO        1618
FRECUENTE    1322
VIP           465
UNKNOWN        23


,cliente_id,customer_id,country,segmento,num_facturas
0,1,13085,United Kingdom,FRECUENTE,8
1,2,13078,United Kingdom,VIP,57
2,3,15362,United Kingdom,REGULAR,2


#### dim_producto
Por cada `StockCode`: descripción más frecuente y precio mediano como referencia.

In [65]:
desc_frecuente = (
    df[df['description'] != 'SIN DESCRIPCION'] # != "diferente de" o "no igual a"
    .groupby('stock_code')['description']
    .agg(lambda x: x.value_counts().index[0]) # evalua las diferentes descripciones de un mismo stock_code
    # .agg reduce muchas filas a un solo resultado
    .reset_index()
    .rename(columns={'description': 'descripcion'})
)
precio_mediano = (
    df[df['is_cancelled'] == 0]
    .groupby('stock_code')['price']
    .median()
    .reset_index()
    .rename(columns={'price': 'precio_referencia'})
)
dim_producto = (
    df[['stock_code']].drop_duplicates()
    .merge(desc_frecuente, on='stock_code', how='left') #merge es JOIN
    .merge(precio_mediano, on='stock_code', how='left')
)
dim_producto['descripcion']         = dim_producto['descripcion'].fillna('SIN DESCRIPCION')
dim_producto['precio_referencia']   = dim_producto['precio_referencia'].fillna(0).round(2)
dim_producto['producto_id']         = range(1, len(dim_producto)+1)
dim_producto                        = dim_producto[['producto_id','stock_code','descripcion','precio_referencia']]

print(f'dim_producto: {len(dim_producto):,} productos')
dim_producto.head(3)

dim_producto: 4,916 productos


,producto_id,stock_code,descripcion,precio_referencia
0,1,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,7.95
1,2,79323P,PINK CHERRY LIGHTS,6.75
2,3,79323W,WHITE CHERRY LIGHTS,6.75


#### dim_geografia
Asigna una region a cada pais para analisis agrupados en el dashboard.

In [66]:
regiones ={
    'United Kingdom':'Europa Occidental','Germany':'Europa Occidental',
    'France':'Europa Occidental','Netherlands':'Europa Occidental',
    'Belgium':'Europa Occidental','Switzerland':'Europa Occidental',
    'Portugal':'Europa Occidental','Spain':'Europa Occidental',
    'Austria':'Europa Occidental','Luxembourg':'Europa Occidental',
    'EIRE':'Europa Occidental','Norway':'Europa Nordica',
    'Sweden':'Europa Nordica','Denmark':'Europa Nordica',
    'Finland':'Europa Nordica','Iceland':'Europa Nordica',
    'Poland':'Europa del Este','Czech Republic':'Europa del Este',
    'Lithuania':'Europa del Este','Italy':'Europa del Sur',
    'Greece':'Europa del Sur','Malta':'Europa del Sur',
    'Cyprus':'Europa del Sur','USA':'America',
    'Canada':'America','Brazil':'America',
    'Australia':'Oceania','Japan':'Asia',
    'Singapore':'Asia','Hong Kong':'Asia',
    'Thailand':'Asia','Israel':'Oriente Medio',
    'Lebanon':'Oriente Medio','Bahrain':'Oriente Medio',
    'Saudi Arabia':'Oriente Medio','United Arab Emirates':'Oriente Medio',
    'South Africa':'Africa','Nigeria':'Africa',
    'RSA':'Africa','Channel Islands':'Otros',
    'European Community':'Otros','Unspecified':'Otros',
    'Bermuda':'America', 'West Indies':'America', 'Korea':'Asia'
}
dim_geografia = df[['country']].drop_duplicates().reset_index(drop=True)
dim_geografia['region'] = dim_geografia['country'].map(regiones).fillna('Otros')
dim_geografia['geografia_id'] = range(1, len(dim_geografia) + 1)
dim_geografia = dim_geografia[['geografia_id', 'country', 'region']]

print(f'dim_geografia: {len(dim_geografia):,} paises')
print('\nDistribucion por region: ')
print(dim_geografia ['region'].value_counts().to_string())
dim_geografia.head(5)

dim_geografia: 43 paises

Distribucion por region: 
region
Europa Occidental    10
Asia                  5
Europa Nordica        5
Oriente Medio         5
America               5
Europa del Sur        4
Europa del Este       3
Otros                 3
Africa                2
Oceania               1


,geografia_id,country,region
0,1,United Kingdom,Europa Occidental
1,2,France,Europa Occidental
2,3,Australia,Oceania
3,4,EIRE,Europa Occidental
4,5,Germany,Europa Occidental


#### fact_ventas
Tabla central, contiene metricas y claves foraneas hacia todas las dimensiones

In [67]:
# =====================================================================
# 1. CREACIÓN DE DICCIONARIOS (TRADUCTORES EN MEMORIA)
# Toma dos columnas (el dato real y su ID) y las empareja como {Llave: Valor} 
# para hacer búsquedas a la velocidad de la luz, sin usar .merge()
# =====================================================================
# zip() empareja dos columnas fila por fila | .dict() convierte esas parejas en un traductor rápido (llave: valor).
fecha_map       = dict(zip(dim_fecha['fecha_completa'].dt.normalize(), dim_fecha['fecha_id']))
cliente_map     = dict(zip(dim_cliente['customer_id'],  dim_cliente['cliente_id']))
producto_map    = dict(zip(dim_producto['stock_code'],  dim_producto['producto_id']))
geo_map         = dict(zip(dim_geografia['country'],    dim_geografia['geografia_id']))

# =====================================================================
# 2. INYECCIÓN DE IDs (EL BUSCAR Y REEMPLAZAR)
# Crea una columna nueva. Toma el dato original, lo busca en el diccionario 
# correspondiente (.map) y guarda su ID numérico (como un BUSCARV de Excel).
# =====================================================================
fact = df.copy()
fact['fecha_id']        = fact['invoice_date'].dt.normalize().map(fecha_map)
fact['cliente_id']      = fact['customer_id'].map(cliente_map)
fact['producto_id']     = fact['stock_code'].map(producto_map)
fact['geografia_id']    = fact['country'].map(geo_map)

# Extraemos la hora entera directamente de la fecha original
fact['hora_venta']  = fact['invoice_date'].dt.hour

# =====================================================================
# 3. CONSTRUCCIÓN DE LA TABLA DE HECHOS (FACT TABLE)
# Filtra la tabla para quedarse SOLO con llaves foráneas (IDs) y métricas 
# (eliminando textos pesados) y traduce los nombres de las columnas al español.
# =====================================================================
fact_ventas = fact[[
    'invoice','fecha_id','hora_venta','cliente_id','producto_id',
    'geografia_id','quantity','price','total_line','is_cancelled'
]].rename(columns={
    'invoice'       :'invoice_number',
    'quantity'      :'cantidad',
    'price'         :'precio_unitario',
    'total_line'    :'total_linea'
})

# =====================================================================
# 4. LIMPIEZA FINAL Y CREACIÓN DE PRIMARY KEY (PK)
# Elimina ventas huérfanas, maneja clientes anónimos (ID 0), convierte  
# las Llaves Foráneas (FK) a enteros limpios y genera un ID transaccional.
# =====================================================================
fact_ventas = fact_ventas.dropna(subset=['fecha_id','producto_id','geografia_id'])
fact_ventas['fecha_id'] = fact_ventas['fecha_id'].astype(int)
fact_ventas['cliente_id'] = fact_ventas['cliente_id'].fillna(0).astype(int)
fact_ventas['producto_id'] = fact_ventas['producto_id'].astype(int)
fact_ventas['geografia_id'] = fact_ventas['geografia_id'].astype(int)

# Aseguramos que la columna sea de tipo entero limpio
fact_ventas['hora_venta'] = fact_ventas['hora_venta'].astype(int)

fact_ventas = fact_ventas.reset_index(drop=True)
fact_ventas.insert(0,'ventas_id', range(1, len(fact_ventas) + 1))

print(f'fact_ventas: {len(fact_ventas):,} filas')
fact_ventas.head(3)

fact_ventas: 1,021,332 filas


,ventas_id,invoice_number,fecha_id,hora_venta,cliente_id,producto_id,geografia_id,cantidad,precio_unitario,total_linea,is_cancelled
0,1,489434,1,7,1,1,1,12,6.95,83.4,0
1,2,489434,1,7,1,2,1,12,6.75,81.0,0
2,3,489434,1,7,1,3,1,12,6.75,81.0,0


### 2.4 Resumen del Transform

In [68]:
print('='*45)
print('RESUMEN DATA WAREHOUSE')
print('='*45)
print(f'dim_fecha       :{len(dim_fecha):>8,} filas')
print(f'dim_cliente     :{len(dim_cliente):>8,} filas')
print(f'dim_producto    :{len(dim_producto):>8,} filas')
print(f'dim_geografia   :{len(dim_geografia):>8,} filas')
print(f'fact_ventas      :{len(fact_ventas):>8,} filas')
print('='*45)
total_revenue = fact_ventas[fact_ventas['is_cancelled']==0]['total_linea'].sum()
print(f'Revenue total: GBP {total_revenue:>12,.2f}')
print(f'Paises : {len(dim_geografia):>8,}')
print(f'Productos : {len(dim_producto):>8,}')
print(f'Clientes: {len(dim_cliente):>8,}')

RESUMEN DATA WAREHOUSE
dim_fecha       :     604 filas
dim_cliente     :   5,876 filas
dim_producto    :   4,916 filas
dim_geografia   :      43 filas
fact_ventas      :1,021,332 filas
Revenue total: GBP 19,645,617.79
Paises :       43
Productos :    4,916
Clientes:    5,876


---
## STEP 3 — LOAD

> **Requisito previo:** crear la base de datos en SSMS antes de ejecutar esta celda:
> ```sql
> CREATE DATABASE OnlineRetail_DW;
> ```

In [69]:
engine = create_engine(CONN_STR, fast_executemany=True)

ddl_statements = [
    "IF OBJECT_ID('fact_ventas',        'U') IS NOT NULL DROP TABLE fact_ventas",
    "IF OBJECT_ID('dim_fecha',          'U') IS NOT NULL DROP TABLE dim_fecha",
    "IF OBJECT_ID('dim_cliente',        'U') IS NOT NULL DROP TABLE dim_cliente",
    "IF OBJECT_ID('dim_producto',       'U') IS NOT NULL DROP TABLE dim_producto",
    "IF OBJECT_ID('dim_geografia',         'U') IS NOT NULL DROP TABLE dim_geografia",
    """CREATE TABLE dim_fecha(
        fecha_id        INT             PRIMARY KEY,
        fecha_completa  DATE            NOT NULL,
        anio            SMALLINT        NOT NULL,
        trimestre       TINYINT         NOT NULL,
        mes             TINYINT         NOT NULL,
        mes_nombre      NVARCHAR(20)    NOT NULL,
        semana          TINYINT         NOT NULL,
        dia             TINYINT         NOT NULL,
        dia_semana      TINYINT         NOT NULL,
        nombre_dia      NVARCHAR(15)    NOT NULL,
        es_fin_semana   BIT             NOT NULL
        )""",
    """CREATE TABLE dim_cliente(
        cliente_id      INT             PRIMARY KEY,
        customer_id     NVARCHAR(20)    NOT NULL,
        country         NVARCHAR(60)    NOT NULL,
        segmento        NVARCHAR(20)    NOT NULL,
        num_facturas    INT             NOT NULL
        )""",
    """CREATE TABLE dim_producto(
        producto_id         INT             PRIMARY KEY,
        stock_code          NVARCHAR(20)    NOT NULL,
        descripcion         NVARCHAR(200)   NOT NULL,
        precio_referencia   DECIMAL(10,2)   NOT NULL
    )""",
    """CREATE TABLE dim_geografia(
        geografia_id    INT             PRIMARY KEY,
        country         NVARCHAR(60)    NOT NULL,
        region          NVARCHAR(40)    NOT NULL
    )""",
    """CREATE TABLE fact_ventas(
        ventas_id       INT             PRIMARY KEY,
        invoice_number  NVARCHAR(20)    NOT NULL,
        fecha_id        INT             NOT NULL    REFERENCES  dim_fecha(fecha_id),
        hora_venta      INT             NOT NULL,
        cliente_id      INT             NOT NULL    REFERENCES  dim_cliente(cliente_id),
        producto_id     INT             NOT NULL    REFERENCES  dim_producto(producto_id),
        geografia_id    INT             NOT NULL    REFERENCES  dim_geografia(geografia_id),
        cantidad        INT             NOT NULL,
        precio_unitario DECIMAL(10,2)   NOT NULL,
        total_linea     DECIMAL(10,2)   NOT NULL,
        is_cancelled    BIT             NOT NULL
    )""",
    "CREATE INDEX IX_fact_fecha     ON  fact_ventas     (fecha_id)",
    "CREATE INDEX IX_fact_cliente   ON  fact_ventas     (cliente_id)",
    "CREATE INDEX IX_fact_producto  ON  fact_ventas     (producto_id)",
    "CREATE INDEX IX_fact_geo       ON  fact_ventas     (geografia_id)",
    "CREATE INDEX IX_fact_cancel    ON  fact_ventas     (is_cancelled)",
    "CREATE INDEX IX_fecha_anio     ON  dim_fecha       (anio,mes)",
    "CREATE INDEX IX_cli_segmento   ON  dim_cliente     (segmento)",
    "CREATE INDEX IX_geo_region     ON  dim_geografia   (region)",
]

print('Creando tablas en SQL Server...')
with engine.connect() as conn:
    for stmt in ddl_statements:
        conn.execute(text(stmt))
    conn.commit()
print('Tablas creadas correctamente.')

Creando tablas en SQL Server...
Tablas creadas correctamente.


### Insertar datos en SQL Server

In [70]:
tablas = [
    ('dim_fecha',       dim_fecha),
    ('dim_cliente',     dim_cliente),
    ('dim_producto',    dim_producto),
    ('dim_geografia',   dim_geografia),
    ('fact_ventas',     fact_ventas),
]

for nombre, df_tabla in tablas:
    print(f' Cargando {nombre:<18} ({len(df_tabla):>9,} filas)...', end=' ')
    df_tabla.to_sql(
        nombre,
        con=engine,
        if_exists='append',
        index=False,
        chunksize=5000,
        # method='multi'
    )
    print('OK')

print('\n' + '=' * 50)
print('ETL COMPLETADO EXITOSAMENTE')
print('=' * 50)
print(f'\n Conecta Power BI: Obtener datos -> SQL Server')
print(f' Servidor: {DB_SERVER} | Base: {DB_NAME}')

 Cargando dim_fecha          (      604 filas)... OK
 Cargando dim_cliente        (    5,876 filas)... OK
 Cargando dim_producto       (    4,916 filas)... OK
 Cargando dim_geografia      (       43 filas)... OK
 Cargando fact_ventas        (1,021,332 filas)... OK

ETL COMPLETADO EXITOSAMENTE

 Conecta Power BI: Obtener datos -> SQL Server
 Servidor: localhost | Base: OnlineRetail_DW
